# AI IncidentOps: Autonomous Multi-Agent IT Incident Response War-Room
### End-to-End Multi-Agent Demonstration Trace

This runnable notebook demonstrates how specialized AI agents collaborate through a centralized **Blackboard Architecture** to investigate and mitigate production IT incidents without hallucinated evidence.

In [ ]:
import os
import sys
from pathlib import Path

# Set project root
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.incident_board import IncidentBoard
from core.orchestrator import IncidentOrchestrator
from infrastructure.simulation_runner import SimulationRunner

print("✓ AI IncidentOps modules successfully imported!")

## 1. Scenario A: Database Connectivity Failure
Simulates an unavailable PostgreSQL instance causing connection pool exhaustion on `payment-api`.

In [ ]:
demo_db = PROJECT_ROOT / "database" / "demo_incidentops.db"
demo_logs = PROJECT_ROOT / "logs"

sim = SimulationRunner(logs_dir=demo_logs)
alert_payload = sim.setup_database_failure()

board = IncidentBoard(demo_db)
orchestrator = IncidentOrchestrator(board, logs_dir=demo_logs)

print(f"[Alert Received] Service: {alert_payload['service']} | Severity: {alert_payload['severity'].upper()}")
print(f"Description: {alert_payload['description']}\n")

# Execute War-Room response cycle
postmortem = await orchestrator.run_incident_response(alert_payload)

print("=== WAR-ROOM EXECUTION TRACE ===")
incident = board.get_incident(postmortem.incident_id)

print(f"Incident ID: {incident.incident_id}")
print("\n--- 1. Logs Agent Evidence ---")
for ev in [e for e in incident.evidence if e.agent == 'Logs Agent']:
    print(f"  [{ev.evidence_id}] {ev.finding}")

print("\n--- 2. Dependency Agent Evidence ---")
for ev in [e for e in incident.evidence if e.agent == 'Dependency Agent']:
    print(f"  [{ev.evidence_id}] {ev.finding}")

print("\n--- 3. Mitigation Agent Proposal ---")
for prop in incident.mitigation_proposals:
    print(f"  [{prop.proposal_id}] ACTION: {prop.proposed_action.upper()} on {prop.target_service} (Risk: {prop.risk_level.upper()})")
    print(f"  Rationale: {prop.reason}")

print("\n--- 4. Incident Commander Final Decision ---")
dec = incident.decision
print(f"  Likely Root Cause: {dec.likely_root_cause}")
print(f"  Mitigation Decision: {dec.mitigation_decision.upper()} on {dec.target_service}")
print(f"  Confidence Rating: {dec.confidence * 100:.1f}%")
print(f"  Supporting Evidence IDs: {', '.join(dec.root_cause_evidence)}")
print(f"  Decision Status: {dec.decision_status.upper()} (Escalation Required: {dec.escalation_required})")

print("\n--- 5. Stakeholder Communication Update ---")
if incident.status_updates:
    print(f"  Headline: {incident.status_updates[-1].headline}")
    print(f"  Broadcast: \"{incident.status_updates[-1].message}\"")

## 2. Scenario D: Insufficient Evidence Handling (Safe Escalation)
Verifies that when telemetry is clean and evidence is ambiguous, the Incident Commander **refuses to invent a root cause** and safely escalates to on-call SREs.

In [ ]:
alert_d = sim.setup_insufficient_evidence()
pm_d = await orchestrator.run_incident_response(alert_d)
inc_d = board.get_incident(pm_d.incident_id)

print(f"Incident ID: {inc_d.incident_id}")
print(f"Commander Root Cause Assessment: '{inc_d.decision.likely_root_cause}'")
print(f"Mitigation Action: '{inc_d.decision.mitigation_decision}'")
print(f"Confidence: {inc_d.decision.confidence * 100:.1f}%")
print(f"Escalation Required: {inc_d.decision.escalation_required}")
print("\n✓ Confirmed: Zero destructive actions proposed. Safe escalation executed.")

## 3. Chronological Append-Only Event Log
Every agent transaction is recorded immutably with timestamp and sequence.

In [ ]:
timeline = board.get_timeline(incident.incident_id)
for ev in timeline:
    print(f"{ev.timestamp} | {ev.event_id} | {ev.agent:<20} | {ev.event_type}")